In [ ]:
import numpy as np
import os
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Dropout,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [ ]:
BASE_PATH = "/content/drive/MyDrive/pluseguard"

NORMALIZED_PATH = os.path.join(
    BASE_PATH,
    "Normalized"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "Models"
)

os.makedirs(MODEL_PATH, exist_ok=True)

print("Normalized data path:", NORMALIZED_PATH)
print("Model path:", MODEL_PATH)

Normalized data path: /content/drive/MyDrive/pluseguard/Normalized
Model path: /content/drive/MyDrive/pluseguard/Models


In [ ]:
X_train = np.load(
    os.path.join(NORMALIZED_PATH, "X_train_z.npy")
)

y_train = np.load(
    os.path.join(NORMALIZED_PATH, "y_train_z.npy")
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (12744, 1000)
y_train: (12744,)


In [ ]:
X_val = np.load(
    os.path.join(NORMALIZED_PATH, "X_val_z.npy")
)

y_val = np.load(
    os.path.join(NORMALIZED_PATH, "y_val_z.npy")
)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_val: (2312, 1000)
y_val: (2312,)


In [ ]:
X_test = np.load(
    os.path.join(NORMALIZED_PATH, "X_test_z.npy")
)

y_test = np.load(
    os.path.join(NORMALIZED_PATH, "y_test_z.npy")
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (2889, 1000)
y_test: (2889,)


In [ ]:
print("===== TRAINING DISTRIBUTION =====")

unique, counts = np.unique(y_train, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")


print("\n===== VALIDATION DISTRIBUTION =====")

unique, counts = np.unique(y_val, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")


print("\n===== TEST DISTRIBUTION =====")

unique, counts = np.unique(y_test, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")

===== TRAINING DISTRIBUTION =====
Class 0: 5744
Class 1: 7000

===== VALIDATION DISTRIBUTION =====
Class 0: 1437
Class 1: 875

===== TEST DISTRIBUTION =====
Class 0: 1795
Class 1: 1094


In [ ]:
print("Training dtype:", X_train.dtype)

print("NaN in training:", np.isnan(X_train).sum())
print("NaN in validation:", np.isnan(X_val).sum())
print("NaN in testing:", np.isnan(X_test).sum())

print("Inf in training:", np.isinf(X_train).sum())
print("Inf in validation:", np.isinf(X_val).sum())
print("Inf in testing:", np.isinf(X_test).sum())

Training dtype: float32
NaN in training: 0
NaN in validation: 0
NaN in testing: 0
Inf in training: 0
Inf in validation: 0
Inf in testing: 0


In [ ]:
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (12744, 1000, 1)
X_val: (2312, 1000, 1)
X_test: (2889, 1000, 1)


In [ ]:
model = Sequential([

    Input(shape=(1000, 1)),

    # Block 1
    Conv1D(
        filters=32,
        kernel_size=7,
        padding="same"
    ),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.20),

    # Block 2
    Conv1D(
        filters=64,
        kernel_size=5,
        padding="same"
    ),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.20),

    # Block 3
    Conv1D(
        filters=128,
        kernel_size=5,
        padding="same"
    ),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),

    # Block 4
    Conv1D(
        filters=256,
        kernel_size=3,
        padding="same"
    ),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),

    # Feature aggregation
    GlobalAveragePooling1D(),

    # Dense layer
    Dense(128, activation="relu"),
    Dropout(0.30),

    # Binary output
    Dense(1, activation="sigmoid")
])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1000, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1000, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1000, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 500, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 500, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 500, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 500, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 500, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 250, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 250, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 250, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 250, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 125, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 125, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 125, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 125, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 125, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 62, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 62, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 185,153 (723.25 KB)

 Trainable params: 184,193 (719.50 KB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Model compiled successfully.")

Model compiled successfully.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

print("Class weights:")
print(class_weights)

Class weights:
{0: 1.1093314763231197, 1: 0.9102857142857143}


In [ ]:
best_model_path = os.path.join(
    MODEL_PATH,
    "Best_1DCNN.keras"
)

early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=12,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    best_model_path,
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=60,
    batch_size=32,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)

Epoch 1/60
399/399 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8275 - auc: 0.8976 - loss: 0.3703
Epoch 1: val_auc improved from None to 0.90063, saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras

Epoch 1: finished saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras
399/399 ━━━━━━━━━━━━━━━━━━━━ 29s 40ms/step - accuracy: 0.8548 - auc: 0.9234 - loss: 0.3269 - val_accuracy: 0.8080 - val_auc: 0.9006 - val_loss: 0.4921 - learning_rate: 0.0010
Epoch 2/60
395/399 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8785 - auc: 0.9412 - loss: 0.2816
Epoch 2: val_auc improved from 0.90063 to 0.91111, saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras

Epoch 2: finished saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8811 - auc: 0.9437 - loss: 0.2764 - val_accuracy: 0.8430 - val_auc: 0.9111 - val_loss: 0.4026 - learning_rate: 0.0010
Epoch 3/60
394/399

In [ ]:
history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=60,
    batch_size=32,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)

Epoch 1/60
394/399 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9201 - auc: 0.9684 - loss: 0.2006
Epoch 1: val_auc improved from 0.92880 to 0.92884, saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras

Epoch 1: finished saving model to /content/drive/MyDrive/pluseguard/Models/Best_1DCNN.keras
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9196 - auc: 0.9690 - loss: 0.1984 - val_accuracy: 0.8720 - val_auc: 0.9288 - val_loss: 0.3225 - learning_rate: 1.5625e-05
Epoch 2/60
398/399 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9201 - auc: 0.9693 - loss: 0.1970
Epoch 2: val_auc did not improve from 0.92884
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9197 - auc: 0.9693 - loss: 0.1969 - val_accuracy: 0.8715 - val_auc: 0.9288 - val_loss: 0.3222 - learning_rate: 1.5625e-05
Epoch 3/60
394/399 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9206 - auc: 0.9691 - loss: 0.1961
Epoch 3: val_auc did not improve from 0.92884
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/

In [ ]:
best_model = tf.keras.models.load_model(
    best_model_path
)

print("Best model loaded successfully.")

Best model loaded successfully.


In [ ]:
test_results = best_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nTest Results:")

for name, value in zip(
    best_model.metrics_names,
    test_results
):
    print(f"{name}: {value:.4f}")

91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8723 - auc: 0.9366 - loss: 0.3122

Test Results:
loss: 0.3122
compile_metrics: 0.8723


In [ ]:
y_probability = best_model.predict(
    X_test,
    verbose=1
).ravel()

threshold = 0.5

y_pred = (
    y_probability >= threshold
).astype(int)

print("Threshold:", threshold)

91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Threshold: 0.5


In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "MI"
        ],
        digits=4
    )
)

              precision    recall  f1-score   support

      Normal     0.8441    0.9744    0.9046      1795
          MI     0.9437    0.7048    0.8069      1094

    accuracy                         0.8723      2889
   macro avg     0.8939    0.8396    0.8557      2889
weighted avg     0.8818    0.8723    0.8676      2889



In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1749   46]
 [ 323  771]]


In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("===== FINAL TEST METRICS =====")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

===== FINAL TEST METRICS =====
Accuracy : 0.8723
Precision: 0.9437
Recall   : 0.7048
F1 Score : 0.8069
ROC-AUC  : 0.9366


In [ ]:
print("===== FINAL TEST RESULTS =====")

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1 Score : {f1 * 100:.2f}%")
print(f"ROC-AUC  : {roc_auc * 100:.2f}%")

===== FINAL TEST RESULTS =====
Accuracy : 87.23%
Precision: 94.37%
Recall   : 70.48%
F1 Score : 80.69%
ROC-AUC  : 93.66%


In [ ]:
final_model_path = os.path.join(
    MODEL_PATH,
    "MI_1DCNN_Final.keras"
)

best_model.save(
    final_model_path
)

print("Final model saved:")
print(final_model_path)

Final model saved:
/content/drive/MyDrive/pluseguard/Models/MI_1DCNN_Final.keras


In [ ]:
print("====================================")
print("       NOTEBOOK 5 COMPLETED")
print("====================================")

print("\nModel:")
print("1D CNN")

print("\nInput:")
print("Lead II ECG")
print("1000 samples")
print("100 Hz")
print("10 seconds")

print("\nClasses:")
print("0 = Normal")
print("1 = MI")

print("\nPreprocessing:")
print("Per-ECG Z-score normalization")

print("\nTraining:")
print("Hardware-noise augmented MI included")

print("\nValidation:")
print("Clean, non-augmented")

print("\nTesting:")
print("Clean, non-augmented")

print("\nFinal Accuracy:")
print(f"{accuracy * 100:.2f}%")

print("\nFinal ROC-AUC:")
print(f"{roc_auc:.4f}")

print("\nModel saved at:")
print(final_model_path)

       NOTEBOOK 5 COMPLETED

Model:
1D CNN

Input:
Lead II ECG
1000 samples
100 Hz
10 seconds

Classes:
0 = Normal
1 = MI

Preprocessing:
Per-ECG Z-score normalization

Training:
Hardware-noise augmented MI included

Validation:
Clean, non-augmented

Testing:
Clean, non-augmented

Final Accuracy:
87.23%

Final ROC-AUC:
0.9366

Model saved at:
/content/drive/MyDrive/pluseguard/Models/MI_1DCNN_Final.keras


In [ ]:
import numpy as np
import os
import tensorflow as tf

BASE_PATH = "/content/drive/MyDrive/pluseguard"
NORMALIZED_PATH = os.path.join(BASE_PATH, "Normalized")
MODEL_PATH = os.path.join(BASE_PATH, "Models")

# Load normalized test ECG data
X_test = np.load(
    os.path.join(NORMALIZED_PATH, "X_test_z.npy")
)

# Load test labels
y_test = np.load(
    os.path.join(NORMALIZED_PATH, "y_test_z.npy")
)

# Add channel dimension for CNN
X_test = X_test[..., np.newaxis]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (2889, 1000, 1)
y_test shape: (2889,)


In [ ]:
# Load final model
model = tf.keras.models.load_model(
    os.path.join(MODEL_PATH, "MI_1DCNN_Final.keras")
)

# Generate sigmoid probabilities
y_probability = model.predict(
    X_test,
    verbose=1
).ravel()

# Separate according to ACTUAL labels
normal_probabilities = y_probability[y_test == 0]
mi_probabilities = y_probability[y_test == 1]

print("===== NORMAL ECG =====")
print("Count:", len(normal_probabilities))
print("Minimum:", np.min(normal_probabilities))
print("Maximum:", np.max(normal_probabilities))
print("Mean:", np.mean(normal_probabilities))
print("Median:", np.median(normal_probabilities))

print("\n===== MI ECG =====")
print("Count:", len(mi_probabilities))
print("Minimum:", np.min(mi_probabilities))
print("Maximum:", np.max(mi_probabilities))
print("Mean:", np.mean(mi_probabilities))
print("Median:", np.median(mi_probabilities))

91/91 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step
===== NORMAL ECG =====
Count: 1795
Minimum: 0.0016543965
Maximum: 0.99538416
Mean: 0.08757367
Median: 0.039242845

===== MI ECG =====
Count: 1094
Minimum: 0.0057232743
Maximum: 1.0
Mean: 0.71819097
Median: 0.9571657


In [ ]:
import pandas as pd
results = pd.DataFrame({
    "Actual_Label": y_test,
    "Sigmoid_Value": y_probability
})

results["Actual_Class"] = results["Actual_Label"].map({
    0: "Normal",
    1: "MI"
})

print(results.head(20))

    Actual_Label  Sigmoid_Value Actual_Class
0              0       0.007511       Normal
1              0       0.053257       Normal
2              0       0.939224       Normal
3              0       0.241768       Normal
4              0       0.063332       Normal
5              1       0.136677           MI
6              0       0.011732       Normal
7              0       0.173435       Normal
8              0       0.164162       Normal
9              0       0.063354       Normal
10             0       0.015810       Normal
11             0       0.028677       Normal
12             0       0.027714       Normal
13             0       0.006970       Normal
14             0       0.040672       Normal
15             0       0.023865       Normal
16             1       0.996491           MI
17             1       0.618818           MI
18             1       0.999986           MI
19             0       0.836939       Normal


In [ ]:
print("===== NORMAL SIGMOID DISTRIBUTION =====")
print(
    np.percentile(
        normal_probabilities,
        [0, 10, 25, 50, 75, 90, 95, 99, 100]
    )
)

print("\n===== MI SIGMOID DISTRIBUTION =====")
print(
    np.percentile(
        mi_probabilities,
        [0, 10, 25, 50, 75, 90, 95, 99, 100]
    )
)

===== NORMAL SIGMOID DISTRIBUTION =====
[0.0016544  0.00852215 0.01653938 0.03924285 0.09514505 0.21145014
 0.31712856 0.76587181 0.99538416]

===== MI SIGMOID DISTRIBUTION =====
[0.00572327 0.11272033 0.3771273  0.95716572 0.99927267 0.99997839
 0.99999821 1.         1.        ]
